# USB firmware uploader

Select the controller, `.ino` payload, and USB port below. The notebook stages a hash-checked temporary copy whose directory matches the sketch name, carries along existing local quoted includes such as `wifi_credentials.h`, compiles with the repository's established board target, and uploads with verification.

It never edits the selected payload and never installs board cores or libraries.

For the Yún, USB updates only the ATmega32U4 `.ino` firmware. It does not deploy `yun_stepper_bridge.py` to the Linux processor; use `provision_yun.sh` over SSH for that separate layer.

## Requirements and target map

- Start Jupyter from this repository directory.
- The uploader automatically prefers the downloaded workspace CLI under `../tools/arduino-cli-*` and `.arduino-build/arduino-cli.yaml`. It falls back to `arduino-cli` on `PATH` when the workspace copy is absent.
- The current workspace toolchain contains ESP32 core `3.3.10`, Arduino AVR core `1.8.8`, Controllino AVR core `3.1.3`, Ethernet `2.0.2`, Adafruit ADS1X15/BusIO, ESP Async WebServer, and Async TCP.
- On another checkout, install those cores/libraries or set `ARDUINO_CLI` and `ARDUINO_CONFIG` to an existing installation.

| `TARGET` | Board target | Default payload |
| --- | --- | --- |
| `controllino` | `CONTROLLINO_Boards:avr:controllino_maxi_automation` | `controllino_ethernet_diagnostic.ino` |
| `esp32` | `esp32:esp32:adafruit_feather_esp32s3_nopsram` | `Flow_management_unit_sch1.ino` |
| `yun` | `arduino:avr:yun` | `limit_switch_palas.ino` |

In [ ]:
from pathlib import Path
import shlex
import sys

repository_root = Path.cwd().resolve()
if not (repository_root / "firmware_upload.py").is_file():
    raise RuntimeError(
        "Start Jupyter in the networked_sensors repository directory "
        "so firmware_upload.py is beside this notebook."
    )
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

from firmware_upload import (
    TARGETS,
    command_preview,
    compile_and_upload,
    discover_ports,
    format_ports,
    get_target,
    monitor_command,
    collect_local_dependencies,
    resolve_payload,
    resolve_toolchain,
    select_port,
    sha256_file,
)

print("Targets:", ", ".join(TARGETS))

## 1. Select the target and payload

Use `PAYLOAD = None` for the target's repository default, or provide another main `.ino` path. Existing local files reached by quoted includes are staged automatically. Use `PORT = "auto"` for exact-FQBN auto-detection. If Arduino CLI cannot identify the board, set the explicit path shown by the port-discovery cell, preferably `/dev/serial/by-id/...` when available.

In [ ]:
TARGET = "esp32"   # "controllino", "esp32", or "yun"
PAYLOAD = None      # e.g. "Flow_management_unit_sch1.ino"
PORT = "auto"      # e.g. "/dev/ttyACM0"
ARDUINO_CLI = None   # auto-detect, or set an executable path
ARDUINO_CONFIG = None  # auto-detect, or set arduino-cli.yaml

# The upload cell remains blocked until you change this after doing the
# physical safety check printed below.
SAFETY_CONFIRMED = False

In [ ]:
profile = get_target(TARGET)
payload_path = resolve_payload(PAYLOAD, target=profile)
toolchain = resolve_toolchain(
    executable=ARDUINO_CLI,
    config_file=ARDUINO_CONFIG,
)
staged_inputs = collect_local_dependencies(payload_path)

print(f"Target:  {profile.label}")
print(f"FQBN:    {profile.fqbn}")
print(f"Payload: {payload_path}")
print(f"SHA-256: {sha256_file(payload_path)}")
print(f"CLI:     {toolchain.executable}")
print(f"Config:  {toolchain.config_file or 'Arduino CLI defaults'}")
print("Inputs:  " + ", ".join(path.name for path in staged_inputs))
print(f"Safety:  {profile.safety_note}")
print("\nCommand shape (temporary paths are filled in during execution):")
print(command_preview(profile, payload_path, PORT, toolchain=toolchain))

## 2. Detect and confirm the USB port

With `PORT = "auto"`, this accepts exactly one port reporting the selected target's exact FQBN. It deliberately refuses to guess when there are zero or multiple matches. An explicit `PORT` overrides matching, so visually confirm the list before proceeding. Stop any dashboard or serial monitor currently holding that port.

In [ ]:
ports = discover_ports(
    executable=toolchain.executable,
    config_file=toolchain.config_file,
)
print(format_ports(ports))
selected_port = select_port(profile, PORT, ports)
print(f"\nSelected upload port: {selected_port}")

## 3. Compile and upload

**Before setting `SAFETY_CONFIRMED = True:**

- ESP32: make connected solenoid loads safe; upload resets the controller. If connection fails, hold **BOOT**, tap **RESET**, release **BOOT**, and retry.
- Yún: set D4 OFF, disconnect the brushless ESC battery, and keep the DM542T motor supply off. Upload resets the ATmega32U4 and the port may re-enumerate.

The cell compiles first, uploads the compiled artifact to the selected USB port, and asks Arduino CLI to verify it. Set `COMPILE_ONLY = True` to test the toolchain without touching a board.

In [ ]:
COMPILE_ONLY = False

result = compile_and_upload(
    TARGET,
    PAYLOAD,
    port=PORT,
    safety_confirmed=SAFETY_CONFIRMED,
    compile_only=COMPILE_ONLY,
    executable=toolchain.executable,
    config_file=toolchain.config_file,
)

action = "Uploaded and verified" if result.uploaded else "Compiled"
print(f"\n{action}: {result.payload.name}")
print(f"Target: {result.target.fqbn}")
print(f"Port: {result.port or 'not used'}")
print("Inputs: " + ", ".join(result.staged_files))
print(f"SHA-256: {result.payload_sha256}")

## 4. Optional serial monitor

The Yún often disconnects and reconnects under another `/dev/ttyACM*` name after upload. Re-run the port-detection cell if necessary, then copy the printed command into a terminal. The monitor runs until you press Ctrl-C.

In [ ]:
MONITOR_PORT = result.port or selected_port
print(shlex.join(monitor_command(TARGET, MONITOR_PORT, toolchain=toolchain)))